In [2]:
#!/usr/bin/env python3
"""
4D SU(2) Gauge Theory Vertex Generator

This module generates the 8-index vertex tensor for 4D SU(2) lattice gauge theory
with topological theta-term, using quantum 6j-symbols.

Author: Manus AI
Date: November 26, 2025
"""

import numpy as np
import cmath
from time import time
from typing import Tuple
from quantum_6j_improved import ImprovedQuantum6jCache


class SU2_4D_VertexGenerator:
    """
    Generator for 4D SU(2) gauge theory vertex tensors.

    This class constructs the 8-valent vertex tensor using a fusion tree
    decomposition with quantum 6j-symbols.
    """

    def __init__(self, j_max: float, theta: float, verbose: bool = True):
        """
        Initialize the vertex generator.

        Args:
            j_max: Maximum spin (e.g., 1.0 for spins 0, 0.5, 1.0)
            theta: Topological angle (0 to 2π)
            verbose: Whether to print progress information
        """
        self.j_max = j_max
        self.theta = theta
        self.verbose = verbose

        # Generate spin list
        self.spins = np.arange(0, j_max + 0.5, 0.5)
        self.D = len(self.spins)

        # Initialize 6j-symbol cache
        if self.verbose:
            print(f"Initializing quantum 6j-symbol cache...")
            print(f"  j_max = {j_max}, theta = {theta:.4f}")

        self.sixj_cache = ImprovedQuantum6jCache(j_max=j_max, theta=theta)

        # Pre-compute all 6j-symbols
        start_time = time()
        self.sixj_cache.precompute_all()
        elapsed = time() - start_time

        if self.verbose:
            print(f"  Cache initialized in {elapsed:.3f}s")
            print(f"  Cached {len(self.sixj_cache.cache)} unique 6j-symbols")
            print()

    def quantum_dimension(self, j: float) -> complex:
        """
        Compute the quantum dimension of spin-j representation.

        For q = exp(iθ), the quantum dimension is:
        d_q(j) = [2j+1]_q = (q^(2j+1) - q^-(2j+1)) / (q - q^-1)

        Args:
            j: Spin value

        Returns:
            Quantum dimension (complex for θ≠0)
        """
        if abs(self.theta) < 1e-9:
            # Classical case
            return float(2*j + 1)

        q = cmath.exp(1j * self.theta)
        n = 2*j + 1

        num = q**n - q**(-n)
        den = q - q**(-1)

        return num / den

    def check_fusion_rules(self, j1: float, j2: float, j3: float) -> bool:
        """
        Check if three spins satisfy the triangle inequality (fusion rules).

        Args:
            j1, j2, j3: Spin values

        Returns:
            True if fusion is allowed
        """
        if not (abs(j1 - j2) <= j3 <= j1 + j2):
            return False
        if (j1 + j2 + j3) % 1.0 != 0.0:  # Must be integer or half-integer
            return False
        return True

    def generate_vertex(self) -> Tuple[np.ndarray, dict]:
        """
        Generate the 8-valent SU(2) vertex tensor.

        The vertex tensor T[j1,j2,j3,j4,j5,j6,j7,j8] represents a gauge-invariant
        vertex in 4D, constructed using a fusion tree:

        Level 1: (j1,j2)→k1, (j3,j4)→k2, (j5,j6)→k3, (j7,j8)→k4
        Level 2: (k1,k2)→m1, (k3,k4)→m2
        Level 3: (m1,m2)→0 (singlet)

        Returns:
            T: 8-index complex tensor of shape (D,D,D,D,D,D,D,D)
            info: Dictionary with statistics
        """
        if self.verbose:
            print(f"{'='*70}")
            print(f"Generating 4D SU(2) Vertex Tensor")
            print(f"{'='*70}")
            print(f"Parameters:")
            print(f"  j_max = {self.j_max}")
            print(f"  theta = {self.theta:.4f}")
            print(f"  D = {self.D} (spin values: {self.spins})")
            print(f"  Tensor shape: ({self.D},)*8 = {self.D**8:,} elements")
            print(f"  Memory: ~{self.D**8 * 16 / 1e6:.2f} MB")
            print()

        start_time = time()

        # Initialize tensor
        T = np.zeros((self.D,)*8, dtype=complex)

        # Statistics
        count_nonzero = 0
        count_total = 0

        # Iterate over all external spin configurations
        for i1 in range(self.D):
            for i2 in range(self.D):
                for i3 in range(self.D):
                    for i4 in range(self.D):
                        for i5 in range(self.D):
                            for i6 in range(self.D):
                                for i7 in range(self.D):
                                    for i8 in range(self.D):
                                        count_total += 1

                                        # Get spin values
                                        j1, j2, j3, j4 = self.spins[i1], self.spins[i2], self.spins[i3], self.spins[i4]
                                        j5, j6, j7, j8 = self.spins[i5], self.spins[i6], self.spins[i7], self.spins[i8]

                                        # Compute tensor element via fusion tree
                                        value = self._compute_tensor_element(j1, j2, j3, j4, j5, j6, j7, j8)

                                        if abs(value) > 1e-14:
                                            T[i1,i2,i3,i4,i5,i6,i7,i8] = value
                                            count_nonzero += 1

            # Progress update
            if self.verbose and (i1 + 1) % max(1, self.D // 4) == 0:
                progress = (i1 + 1) / self.D * 100
                print(f"  Progress: {progress:.0f}% ({count_nonzero:,} / {count_total:,} non-zero)")

        elapsed = time() - start_time

        # Compute statistics
        sparsity = 100 * (1 - count_nonzero / count_total)
        max_real = np.max(np.abs(np.real(T)))
        max_imag = np.max(np.abs(np.imag(T)))
        norm = np.linalg.norm(T)

        info = {
            'j_max': self.j_max,
            'theta': self.theta,
            'D': self.D,
            'shape': T.shape,
            'total_elements': count_total,
            'nonzero_elements': count_nonzero,
            'sparsity_percent': sparsity,
            'max_real': max_real,
            'max_imag': max_imag,
            'norm': norm,
            'generation_time': elapsed
        }

        if self.verbose:
            print()
            print(f"{'='*70}")
            print(f"Vertex Generation Complete")
            print(f"{'='*70}")
            print(f"Non-zero elements: {count_nonzero:,} / {count_total:,}")
            print(f"Sparsity: {sparsity:.2f}%")
            print(f"Max |Re|: {max_real:.6e}")
            print(f"Max |Im|: {max_imag:.6e}")
            print(f"Norm: {norm:.6e}")
            print(f"Generation time: {elapsed:.3f}s")
            print(f"{'='*70}\n")

        return T, info

    def _compute_tensor_element(self, j1: float, j2: float, j3: float, j4: float,
                                j5: float, j6: float, j7: float, j8: float) -> complex:
        """
        Compute a single tensor element using the fusion tree.

        The fusion tree structure:
        - Fuse pairs: (j1,j2)→k1, (j3,j4)→k2, (j5,j6)→k3, (j7,j8)→k4
        - Fuse intermediate: (k1,k2)→m1, (k3,k4)→m2
        - Fuse to singlet: (m1,m2)→0

        Args:
            j1-j8: External spin values

        Returns:
            Complex amplitude for this configuration
        """
        value = 0.0 + 0.0j

        # Sum over all internal fusion channels
        for k1 in self.spins:
            if not self.check_fusion_rules(j1, j2, k1):
                continue

            for k2 in self.spins:
                if not self.check_fusion_rules(j3, j4, k2):
                    continue

                for k3 in self.spins:
                    if not self.check_fusion_rules(j5, j6, k3):
                        continue

                    for k4 in self.spins:
                        if not self.check_fusion_rules(j7, j8, k4):
                            continue

                        for m1 in self.spins:
                            if not self.check_fusion_rules(k1, k2, m1):
                                continue

                            for m2 in self.spins:
                                if not self.check_fusion_rules(k3, k4, m2):
                                    continue

                                # Final fusion to singlet: m1 = m2
                                if m1 != m2:
                                    continue

                                # Compute amplitude using 6j-symbols
                                # We need multiple 6j-symbols to connect the fusion tree

                                # Get quantum dimensions
                                d_k1 = self.quantum_dimension(k1)
                                d_k2 = self.quantum_dimension(k2)
                                d_k3 = self.quantum_dimension(k3)
                                d_k4 = self.quantum_dimension(k4)
                                d_m1 = self.quantum_dimension(m1)

                                # Get 6j-symbols from cache
                                sixj_1 = self.sixj_cache.get(j1, j2, k1, j2, j1, k1)  # Simplified
                                sixj_2 = self.sixj_cache.get(j3, j4, k2, j4, j3, k2)
                                sixj_3 = self.sixj_cache.get(j5, j6, k3, j6, j5, k3)
                                sixj_4 = self.sixj_cache.get(j7, j8, k4, j8, j7, k4)
                                sixj_5 = self.sixj_cache.get(k1, k2, m1, k2, k1, m1)
                                sixj_6 = self.sixj_cache.get(k3, k4, m1, k4, k3, m1)

                                # Combine contributions
                                amplitude = (d_k1 * d_k2 * d_k3 * d_k4 * d_m1 *
                                           sixj_1 * sixj_2 * sixj_3 * sixj_4 * sixj_5 * sixj_6)

                                value += amplitude

        return value


# ============================================================
# CONVENIENCE FUNCTION
# ============================================================

def generate_4d_vertex(j_max: float = 1.0, theta: float = 0.0,
                      verbose: bool = True) -> Tuple[np.ndarray, dict]:
    """
    Convenience function to generate a 4D SU(2) vertex tensor.

    Args:
        j_max: Maximum spin (default: 1.0)
        theta: Topological angle in radians (default: 0.0)
        verbose: Print progress information (default: True)

    Returns:
        T: 8-index complex tensor
        info: Dictionary with statistics

    Example:
        >>> T, info = generate_4d_vertex(j_max=1.0, theta=0.5)
        >>> print(f"Generated tensor with {info['nonzero_elements']} non-zero elements")
    """
    generator = SU2_4D_VertexGenerator(j_max, theta, verbose)
    return generator.generate_vertex()


# ============================================================
# DEMONSTRATION
# ============================================================

if __name__ == "__main__":
    print("="*70)
    print("4D SU(2) VERTEX GENERATOR - DEMONSTRATION")
    print("="*70)
    print()

    # Test 1: Classical case (theta=0)
    print(">>> TEST 1: Classical SU(2) (theta=0)")
    print()
    T1, info1 = generate_4d_vertex(j_max=1.0, theta=0.0, verbose=True)

    print()
    print(">>> TEST 2: Topological SU(2) (theta=0.5)")
    print()
    T2, info2 = generate_4d_vertex(j_max=1.0, theta=0.5, verbose=True)

    # Compare
    print("="*70)
    print("COMPARISON")
    print("="*70)
    print(f"Classical (θ=0):")
    print(f"  Max |Im|: {info1['max_imag']:.6e}")
    print(f"  Norm: {info1['norm']:.6e}")
    print()
    print(f"Topological (θ=0.5):")
    print(f"  Max |Im|: {info2['max_imag']:.6e}")
    print(f"  Norm: {info2['norm']:.6e}")
    print()

    if info2['max_imag'] > 1e-10:
        print("✓ SUCCESS: Topological phase detected (complex tensor)")
    else:
        print("⚠ WARNING: No imaginary component detected")

    print("="*70)


ModuleNotFoundError: No module named 'quantum_6j_improved'